# Phase 2b — Fine-tuning Bio_ClinicalBERT for Neurocognitive Outcome Detection**What this notebook does:**1. Loads your hand-labeled training data (1,804 trials, 4 cancer types)2. Fine-tunes Bio_ClinicalBERT to predict whether a trial measures a neurocognitive outcome3. Evaluates on the SAME held-out test split logic as the R baseline, so results are directly comparable4. Reports the same metrics, broken down by cancer type, plus the keyword-only baseline for comparison**Before you start:** Go to `Runtime` -> `Change runtime type` -> select **T4 GPU** -> Save.This gives you free GPU access, which fine-tuning needs (it will be extremely slow on CPU).**How to run:** Click each cell in order (Shift+Enter), or `Runtime` -> `Run all`.The first cell will ask you to upload `CLASSIFIER_training_population.csv` — use the file I gave you earlier.

## 1. Install packagesThis takes ~1-2 minutes the first time.

In [ ]:
!pip install -q transformers datasets torch scikit-learn pandas accelerate

## 2. Upload your training dataA file-upload button will appear below. Select `CLASSIFIER_training_population.csv`.

In [ ]:
from google.colab import filesuploaded = files.upload()import pandas as pdfname = list(uploaded.keys())[0]df = pd.read_csv(fname)print(f"Loaded {len(df)} rows")df.head(3)

## 3. Prepare the dataSame logic as the R script: build a label column, a keyword-hit flag, and a stratified80/20 train/test split by cancer type + label -- so this test set plays the same roleas the one in your R baseline, and results are genuinely comparable.

In [ ]:
import numpy as npdf = df.dropna(subset=['Outcome text (for cognition check)']).reset_index(drop=True)df['label'] = (df['Measures cognition? (Y/N)'] == 'Yes').astype(int)df['keyword_hit'] = df['Flagged Keyword(s)'].notna().astype(int)df['text'] = df['Outcome text (for cognition check)'].astype(str)df['strata'] = df['Cancer Type'] + '_' + df['label'].astype(str)from sklearn.model_selection import train_test_splittrain_df, test_df = train_test_split(df, test_size=0.2, stratify=df['strata'], random_state=2026)print(f"Train: {len(train_df)}  |  Test (held out): {len(test_df)}")print(train_df.groupby('Cancer Type')['label'].agg(['sum','count']))

## 4. Load Bio_ClinicalBERT and tokenize`max_length=512` is BERT's hard limit -- longer outcome-text trials get truncated tothe first 512 tokens. Most of your trials are well under this; the very long ones(some 30,000+ characters) will lose their tail end. This is a real, honest limitationworth noting -- flagged in the write-up cell at the end.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassificationimport torchMODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)device = torch.device("cuda" if torch.cuda.is_available() else "cpu")print("Using device:", device)model.to(device)

In [ ]:
from datasets import Datasetdef to_hf_dataset(d):    return Dataset.from_pandas(d[['text','label']].reset_index(drop=True))train_ds = to_hf_dataset(train_df)test_ds  = to_hf_dataset(test_df)def tokenize_fn(batch):    return tokenizer(batch['text'], truncation=True, padding='max_length', max_length=512)train_ds = train_ds.map(tokenize_fn, batched=True)test_ds  = test_ds.map(tokenize_fn, batched=True)train_ds.set_format(type='torch', columns=['input_ids','attention_mask','label'])test_ds.set_format(type='torch', columns=['input_ids','attention_mask','label'])

## 5. Fine-tune3 epochs is a reasonable starting point for a dataset this size -- enough to adapt themodel without badly overfitting. This should take roughly 10-20 minutes on a T4 GPU.

In [ ]:
from transformers import TrainingArguments, Trainerfrom sklearn.metrics import accuracy_score, roc_auc_score, f1_scoredef compute_metrics(eval_pred):    logits, labels = eval_pred    probs = torch.softmax(torch.tensor(logits), dim=1)[:,1].numpy()    preds = (probs >= 0.5).astype(int)    return {        'accuracy': accuracy_score(labels, preds),        'auc': roc_auc_score(labels, probs),        'f1': f1_score(labels, preds)    }args = TrainingArguments(    output_dir="./bert_checkpoints",    num_train_epochs=3,    per_device_train_batch_size=8,    per_device_eval_batch_size=16,    learning_rate=2e-5,    eval_strategy="epoch",    save_strategy="epoch",    load_best_model_at_end=True,    metric_for_best_model="auc",    logging_steps=20,    report_to="none",)trainer = Trainer(    model=model, args=args,    train_dataset=train_ds, eval_dataset=test_ds,    compute_metrics=compute_metrics,)trainer.train()

## 6. Evaluate — same metrics, same per-cancer-type breakdown, same baseline comparison as the R script

In [ ]:
preds_output = trainer.predict(test_ds)probs = torch.softmax(torch.tensor(preds_output.predictions), dim=1)[:,1].numpy()preds = (probs >= 0.5).astype(int)labels = test_df['label'].valuesfrom sklearn.metrics import confusion_matrix, roc_auc_score, f1_scorecm = confusion_matrix(labels, preds)tn, fp, fn, tp = cm.ravel()sens = tp/(tp+fn); spec = tn/(tn+fp); ppv = tp/(tp+fp) if (tp+fp)>0 else 0npv = tn/(tn+fn) if (tn+fn)>0 else 0f1 = f1_score(labels, preds)auc = roc_auc_score(labels, probs)print("=== OVERALL held-out test performance ===")print(f"N = {len(labels)}")print(cm)print(f"Sensitivity: {sens:.3f}  Specificity: {spec:.3f}  PPV: {ppv:.3f}  NPV: {npv:.3f}")print(f"F1: {f1:.3f}  AUROC: {auc:.3f}")

In [ ]:
# By cancer type, WITH the naive keyword-only baseline for direct comparisonresults = test_df.copy()results['pred'] = predsresults['prob'] = probsprint(f"{'Type':<10}{'N':>5}{'Sens':>8}{'Spec':>8}{'Acc':>8}{'Baseline':>10}{'Lift':>8}")for ct, g in results.groupby('Cancer Type'):    macc = (g['pred'] == g['label']).mean()    bacc = (g['keyword_hit'] == g['label']).mean()    s = ((g['pred']==1) & (g['label']==1)).sum() / max((g['label']==1).sum(), 1)    sp = ((g['pred']==0) & (g['label']==0)).sum() / max((g['label']==0).sum(), 1)    print(f"{ct:<10}{len(g):>5}{s:>8.1%}{sp:>8.1%}{macc:>8.1%}{bacc:>10.1%}{macc-bacc:>8.1%}")print()print("Compare the 'Lift' column here to the R baseline's lift_over_baseline (which was 0.000")print("everywhere). Real, positive lift here means BERT is genuinely learning something the")print("simple keyword flag could not.")

## 7. Save the fine-tuned modelDownload this and keep it — it's what Phase 4 will use to score the full ~50,000-trial landscape.

In [ ]:
model.save_pretrained("./cognitive_classifier_bert")tokenizer.save_pretrained("./cognitive_classifier_bert")import shutilshutil.make_archive("cognitive_classifier_bert", 'zip', "./cognitive_classifier_bert")files.download("cognitive_classifier_bert.zip")

## Honest notes for your Methods section- **Base model:** Bio_ClinicalBERT (`emilyalsentzer/Bio_ClinicalBERT`), a BERT model further  pre-trained on MIMIC-III clinical notes on top of BioBERT.- **Fine-tuning:** 3 epochs, learning rate 2e-5, batch size 8, on the same 1,804-trial  gold-standard training population used for the TF-IDF baseline.- **Sequence length limit:** inputs truncated to 512 tokens (BERT's architectural limit).  A small number of trials with very long outcome-text lists (multi-thousand character  entries, mostly pediatric CNS trials with many outcome measures) will have their later  outcomes cut off. This is a genuine, disclosed limitation, not an error.- **Comparison:** this notebook uses the identical stratified 80/20 train/test split logic  (by cancer type + label) as the R TF-IDF baseline, so the two are directly comparable —  the "Lift" numbers above are the real answer to whether contextual embeddings add value  beyond the naive keyword rule.